# Simple Pad — Palace Eigenmode and EPR

Canonical source-only candidate; semantic state CONVERGING, delivery
LOCAL CANDIDATE, no test writes. OrPen owns physical source facts, this
notebook owns case settings and final Nets, and SCGSim owns runtime.

This synthetic `square_pad_capacitor` linear LC model uses a 200 µm pad,
20 µm gap and 1000 µm die with current PDK 100 nm Al above 500 µm Si.
Its nonmetal east locator spans x=99.999–120.001, y=-1–1 µm and binds a
linear 10 nH inductive branch, with no added capacitance. This is
distinct from ordinary CPW, generic RLC and a measured device.

Read the [eigenmode
requirements](../../../simulation/requirements/eigenmode.qmd) and
[EPR/loss requirements](../../../simulation/requirements/epr-loss.qmd).
The authoring target is [SCGSim v2
RC1](https://github.com/OrPenStrike/scgsim/tree/85d8bedb733b1e17737d935029618351b30ea5de)
in a separate consumer environment; ordinary project dependencies and
pins remain unchanged.

The default `AUTHORING_ONLY` action constructs the source coupon and
detached Plan/Sim inputs only. It never meshes, generates config,
prepares a native handoff or solves. Heavy preparation requires explicit
`prepare_handoff` or `run`. Quarto execution remains disabled.
LiteratureCentral MA/MS/SA films are synthetic assumptions; numerical
controls are inputs, not adequacy Gates. No new native geometry, mesh,
fields, solve or numerical-accuracy claim is made.

Historical source cohort: OrPen base `36b6e6b`, SCGSim runtime 1.0.1.
Its outputs retain their original run/runtime identities and are not
results of this current source.

In [ ]:
import importlib.metadata
import json
import os
import subprocess
from pathlib import Path
from uuid import uuid4

from IPython.display import display
from scgsim.palace import reanalyze_epr, resolve_epr_result, show_epr

WORKFLOW_ACTION = "AUTHORING_ONLY"
# AUTHORING_ONLY | prepare_handoff | run | analyze_handoff | save_from_returned | analyze_saved_epr
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    import gdsfactory as gf
    from scgsim.geometry import GeometryPlan
    from scgsim.palace import EigenmodeSim

    import orpen_sc_pdk
    from orpen_sc_pdk import (
        get_interface_preset_records,
        get_material_records,
        validate_interface_preset_records,
    )
    from orpen_sc_pdk.tech import get_single_die_layer_stack

if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    from scgsim.palace import inspect_run_trustworthiness, resolve_palace_result

RUN_ID = f"simple_pad_palace_eigenmode_v2_{uuid4().hex}"
OUTPUT_ROOT = Path.cwd() / ".artifacts" / "SimplePad"
RUN_ROOT = OUTPUT_ROOT / RUN_ID
if WORKFLOW_ACTION not in {
    "AUTHORING_ONLY",
    "prepare_handoff",
    "run",
    "analyze_handoff",
    "save_from_returned",
    "analyze_saved_epr",
}:
    raise ValueError("Unknown WORKFLOW_ACTION")
display({"scgsim_version": importlib.metadata.version("scgsim"), "run_id": RUN_ID})

## Build Component Coupon

In [ ]:
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    PAD_SIDE_UM = 200.0
    GAP_UM = 20.0
    DIE_SIDE_UM = 1000.0
    JUNCTION_WIDTH_UM = 2.0
    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(
        "square_pad_capacitor",
        pad_side_um=PAD_SIDE_UM,
        gap_um=GAP_UM,
        die_side_um=DIE_SIDE_UM,
        junction_width_um=JUNCTION_WIDTH_UM,
    )
    layer_stack = get_single_die_layer_stack()
    plan = GeometryPlan(
        component=component,
        root_id="chip",
        layer_stack=layer_stack,
        material_records=get_material_records(),
        coupon_padding_um=0.0,
    )
    plan.set_nets({"pad": ("chip/PAD",), "ground": ("chip/GROUND",)})
    plan.set_vacuum_region(padding=(250.0, 250.0, 1000.0))
    snapshot = plan.prepare()
    display(component)
    display({"bbox_um": str(component.dbbox()), "ports": component.ports})

## Configure EPR / Problem

In [ ]:
if WORKFLOW_ACTION in {"AUTHORING_ONLY", "prepare_handoff", "run"}:
    sim = EigenmodeSim()
    sim.set_plan(snapshot)
    sim.set_output_dir(RUN_ROOT)
    ROUTE = "B"
    records = validate_interface_preset_records(get_interface_preset_records())
    preset_names = {kind: f"LiteratureCentral_{kind}" for kind in ("MA", "MS", "SA")}
    interface_presets = {kind: records[name] for kind, name in preset_names.items()}
    sim.set_surface_epr(
        representation=ROUTE,
        specs={
            kind: {
                "thickness": record["thickness"],
                "permittivity": record["permittivity"],
                "loss_tangent": record["loss_tangent"],
                "source": record["source"],
                "preset": preset_names[kind],
            }
            for kind, record in interface_presets.items()
        },
    )
    PORT_NAME = "chip/o_junction_lumped"
    PORT_TARGET_LAYER = "D0_TOP_M1"
    PORT_INDUCTANCE_H = 10e-9
    PORT_CAPACITANCE_F = 0.0  # Inductive-only model; Palace add_port has no C field.
    NUM_MODES = 1
    SEARCH_FREQUENCY_HZ = 1e9
    EIGENMODE_TOLERANCE = 1e-6
    SAVE_FIELDS = 1
    sim.add_port(
        PORT_NAME, layer=PORT_TARGET_LAYER, layout_sheet=True, inductance=PORT_INDUCTANCE_H
    )
    sim.set_epr_request(
        surface_interfaces=("MA", "MS", "SA"), bulk_domain_ids=None, port_names=(PORT_NAME,)
    )
    sim.set_eigenmode(
        num_modes=NUM_MODES,
        target=SEARCH_FREQUENCY_HZ,
        tolerance=EIGENMODE_TOLERANCE,
        save=SAVE_FIELDS,
    )
    display(
        {
            "route": ROUTE,
            "port": PORT_NAME,
            "target_layer": PORT_TARGET_LAYER,
            "inductance_h": PORT_INDUCTANCE_H,
            "capacitance_f": PORT_CAPACITANCE_F,
            "num_modes": NUM_MODES,
            "search_frequency_hz": SEARCH_FREQUENCY_HZ,
            "eigenmode_tolerance": EIGENMODE_TOLERANCE,
            "save_fields": SAVE_FIELDS,
            "interface_presets": interface_presets,
        }
    )
    display(sim.input_summary())

## Build Mesh

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    REFINED_MESH_SIZE_UM = 10.0
    MAX_MESH_SIZE_UM = 200.0
    # SCGSim owns Delaunay Mesh.Algorithm3D=1; no notebook mesher override.
    ALGORITHM_3D = "Delaunay"
    MESH_THREADS = 1
    SURFACE_THREADS = 1
    GEOMETRY_ORDER = 1  # Geometry mesh degree, not field FEM order.
    HIGH_ORDER_OPTIMIZE = True
    sim.set_mesh(
        refined_mesh_size=REFINED_MESH_SIZE_UM,
        max_mesh_size=MAX_MESH_SIZE_UM,
        algorithm_3d=ALGORITHM_3D,
        threads=MESH_THREADS,
        surface_threads=SURFACE_THREADS,
        geometry_order=GEOMETRY_ORDER,
        high_order_optimize=HIGH_ORDER_OPTIMIZE,
    )
    MESH_PATH = sim.mesh()
    display(MESH_PATH)

## Generate Config

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    FEM_ORDER = 2
    LINEAR_TOLERANCE = 1e-6
    MAX_ITERATIONS = 2000
    AMR_MAX_PASSES = 10
    AMR_TOLERANCE = 0.02
    AMR_UPDATE_FRACTION = 0.15
    AMR_NONCONFORMAL = False
    SAVE_ADAPT_ITERATIONS = True
    sim.set_numerical(
        order=FEM_ORDER,
        tolerance=LINEAR_TOLERANCE,
        max_iterations=MAX_ITERATIONS,
        solver_type="Default",
        preconditioner="Default",
        device="CPU",
        amr_max_passes=AMR_MAX_PASSES,
        amr_tolerance=AMR_TOLERANCE,
        amr_update_fraction=AMR_UPDATE_FRACTION,
        amr_nonconformal=AMR_NONCONFORMAL,
        save_adapt_iterations=SAVE_ADAPT_ITERATIONS,
        output_paraview=True,
        output_grid_function=False,
    )
    CONFIG_PATH = sim.write_config()
    display(CONFIG_PATH)
    display(sim.mesh_summary())

## Prepare Handoff

Preparation performs real local geometry and meshing, but does not start
Palace. Only the explicit `run` action invokes the generated handoff
script. Retain its handoff ID independently of the returned package.
Select an actual MPI wrapper, not a direct ELF: the generated script
supplies `-np 8 -nt 1`. The native executable must include corrected
Palace source `4e9e1b7232b4c8ca8b964be7e70f1848f578de37`; this notebook
does not install it.

In [ ]:
if WORKFLOW_ACTION in {"prepare_handoff", "run"}:
    MACHINE_PROFILE = "direct-local"
    PALACE_EXECUTABLE = os.environ["PALACE_EXECUTABLE"]
    SETUP_COMMANDS = tuple(os.environ.get("PALACE_SETUP_COMMANDS", "").splitlines())
    RESOURCES = {"processes": 8, "threads": 1, "command_style": "wrapper"}
    HANDOFF = sim.prepare_handoff(
        profile=MACHINE_PROFILE,
        executable=PALACE_EXECUTABLE,
        resources=RESOURCES,
        setup_commands=SETUP_COMMANDS,
    )
    display(HANDOFF)
    display({"retain_independently": HANDOFF.handoff_id})
    if WORKFLOW_ACTION == "run":
        subprocess.run([str(HANDOFF.script_path)], cwd=HANDOFF.run_dir, check=True)

## Analyze Returned Run

Start a fresh analysis kernel. Supply the actual returned directory and
the independently retained preparation ID through the named environment
inputs; missing inputs fail natively. Partial inspection is evidence
only and does not substitute for strict completed-result resolution.

### Run Identity

### Numerical Evidence

### Simulation Benchmark

In [ ]:
if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    RETURNED_RUN_DIR = Path(os.environ["SIMPLE_PAD_RETURNED_RUN_DIR"])
    EXPECTED_HANDOFF_ID = os.environ["SIMPLE_PAD_PALACE_HANDOFF_ID"]
    report = inspect_run_trustworthiness(RETURNED_RUN_DIR)
    display(report.show_run_trustworthiness(show_details=True))
    display(report.show_simulation_benchmark(show_details=True))
    if WORKFLOW_ACTION == "save_from_returned" or report.completeness == "complete":
        result = resolve_palace_result(RETURNED_RUN_DIR, expected_handoff_id=EXPECTED_HANDOFF_ID)

### Physics Analysis Results

In [ ]:
if WORKFLOW_ACTION in {"analyze_handoff", "save_from_returned"}:
    if report.completeness == "complete":
        display(result.show_physics_quantities(ranking_limit=None))
        epr = result.epr_result()
        display(show_epr(epr, mode=1))
    else:
        display(report.show_physics_quantities(ranking_limit=None))

### Saved EPR and Offline Loss

Only a strict completed eigenmode result produces this portable
snapshot. Offline changes below retain native participation and source
bindings; they change model film assumptions, not fields. Signed Palace
junction participation stays signed and is never absolute-valued or
labeled negative physical energy. The explicit save path is caller-owned
and opened exclusively to avoid overwrite.

In [ ]:
if WORKFLOW_ACTION == "save_from_returned":
    EPR_SAVE_PATH = Path(os.environ["SIMPLE_PAD_PALACE_EPR_SAVE_PATH"])
    epr = result.epr_result()
    with EPR_SAVE_PATH.open("x", encoding="utf-8") as stream:
        json.dump(epr.to_payload(), stream, indent=2)

Set `WORKFLOW_ACTION = "analyze_saved_epr"` in a fresh kernel to reuse a
specified snapshot. This action reads only the saved EPR path; it
requires no returned run directory, handoff ID, component, GeometryPlan,
mesh or fields. `save_from_returned` creates a new path exclusively from
an actual strict returned result. Existing files are never silently
reused as a save fallback. `analyze_handoff` does not create a snapshot.

In [ ]:
if WORKFLOW_ACTION in {"save_from_returned", "analyze_saved_epr"}:
    EPR_SAVE_PATH = Path(os.environ["SIMPLE_PAD_PALACE_EPR_SAVE_PATH"])
    saved_epr = resolve_epr_result(EPR_SAVE_PATH)
    SURFACE_DEFAULTS = {}  # Caller may explicitly supply MA/MS/SA film assumptions.
    GROUP_OVERRIDES = {}  # Use real Group IDs from saved_epr; never fabricate IDs.
    offline_epr = reanalyze_epr(
        saved_epr,
        surface_defaults=SURFACE_DEFAULTS,
        group_overrides=GROUP_OVERRIDES,
    )
    display(show_epr(offline_epr, mode=1))

### Simulation Performance / Benchmarks

Read the report’s native pass history, mesh/DoF, time and resource
evidence beside the actual requested controls. Missing channels stay
unavailable; no trend, numerical accuracy, device loss or lifetime is
inferred from execution success. Native observations and their
interpretation are pending owner execution.